# Hay-inspired single compartment: dataset and neural surrogates

This notebook generates a reproducible random-input dataset containing the complete Markov state of one challenging compartment, then compares a memoryless MLP with GRU and LSTM surrogates. It is designed to run end-to-end on Kaggle.

In [ ]:
from pathlib import Path
import subprocess, sys

def find_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    candidates += [p.parent for p in Path('/kaggle/working').glob('*/pyproject.toml')] if Path('/kaggle/working').exists() else []
    for candidate in candidates:
        pyproject = candidate / 'pyproject.toml'
        if pyproject.exists() and 'hay-single-compartment' in pyproject.read_text():
            return candidate
    return None

ROOT = find_root()
if ROOT is None:
    ROOT = Path('/kaggle/working/LearningSingleCompartiment')
    if (ROOT / '.git').exists():
        subprocess.check_call(['git', '-C', str(ROOT), 'fetch', 'origin', 'main'])
        subprocess.check_call(['git', '-C', str(ROOT), 'checkout', '-B', 'main', 'origin/main'])
    else:
        subprocess.check_call(['git', 'clone', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(ROOT)])
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
assert (SRC / 'hay_single_compartment').is_dir(), f'Package source missing: {SRC}'
print('Project root:', ROOT)
print('Package source:', SRC)

In [ ]:
import json
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from hay_single_compartment import (
    CURRENT_NAMES, INPUT_NAMES, STATE_NAMES, SimulationConfig,
    generate_dataset, validate_dataset,
)
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_trajectory, train_architectures

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT = Path('/kaggle/working/hay_single_results') if Path('/kaggle').exists() else ROOT / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
DATASET = OUTPUT / 'single_compartment.h5'
print('Device:', DEVICE)

## 1. Generate and validate a small dataset

Seeds are disjoint across whole trajectories. The observed step is 0.1 ms and the internal stable integrator uses 0.025 ms.

In [ ]:
config = SimulationConfig(
    duration_ms=300.0, warmup_ms=100.0, seed=2026,
    train_trajectories=8, validation_trajectories=2, test_trajectories=2,
)
if not DATASET.exists():
    generation_report = generate_dataset(DATASET, config)
else:
    generation_report = validate_dataset(DATASET)
generation_report

In [ ]:
with h5py.File(DATASET, 'r') as h5:
    time_ms = h5['time_ms'][...]
    states = h5['train/states'][0]
    inputs = h5['train/inputs'][0]
    spikes = h5['train/spikes'][0]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(time_ms, states[:, 0], lw=0.8); axes[0].set_ylabel('V (mV)')
axes[1].plot(time_ms[:-1], inputs[:, 0], lw=0.8); axes[1].set_ylabel('I inj (nA)')
axes[2].plot(time_ms, states[:, 1] * 1e3, lw=0.8); axes[2].set_ylabel('Ca i (uM)')
axes[3].plot(time_ms, states[:, 14:17]); axes[3].legend(['AMPA', 'NMDA', 'GABA']); axes[3].set_ylabel('g (uS)')
axes[3].set_xlabel('time (ms)')
fig.suptitle(f'Complete state example — detected spikes: {int(spikes.sum())}')
plt.tight_layout()

## 2. Train matched MLP, GRU, LSTM, and wide ConvLSTM baselines

All models see the current full state plus the known external input and predict the next full state. Normalization uses the train split only. Validation selects the checkpoint; test is touched once at the end. The 950k-parameter ConvLSTM receives 20 epochs while the compact baselines receive 8, because its wider causal convolutional front-end converges more slowly.

In [ ]:
reports = train_architectures(
    DATASET, OUTPUT / 'models', architectures=('mlp', 'gru', 'lstm'),
    epochs=8, sequence_length=64, stride=32, batch_size=64,
    hidden_dim=96, layers=2, device=DEVICE, seed=2026,
)
reports += train_architectures(
    DATASET, OUTPUT / 'models', architectures=('conv_lstm',),
    epochs=20, sequence_length=64, stride=32, batch_size=64,
    hidden_dim=96, layers=2, device=DEVICE, seed=2026,
)
comparison = pd.DataFrame([{
    'model': report['architecture'],
    'parameters': report['parameters'],
    'validation_loss': report['best_validation_loss'],
    'test_voltage_rmse_mV': report['test']['voltage_rmse_mv'],
    'test_mean_normalized_rmse': report['test']['mean_normalized_rmse'],
    'persistence_voltage_rmse_mV': report['test']['persistence_voltage_rmse_mv'],
} for report in reports]).sort_values('test_voltage_rmse_mV')
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for report in reports:
    history = pd.DataFrame(report['history'])
    axes[0].plot(history.epoch, history.train_loss, label=report['architecture'])
    axes[1].plot(history.epoch, history.validation_loss, label=report['architecture'])
axes[0].set(title='Train', xlabel='epoch', ylabel='weighted MSE')
axes[1].set(title='Validation', xlabel='epoch', ylabel='weighted MSE')
for ax in axes: ax.set_yscale('log'); ax.legend(); ax.grid(alpha=.2)
plt.tight_layout()

## 3. Autoregressive rollout

One-step scores can hide compounding error. The following comparison feeds each predicted state back into the best model while retaining only the real future inputs.

In [ ]:
best_name = comparison.iloc[0]['model']
checkpoint = torch.load(OUTPUT / 'models' / f'{best_name}.pt', map_location=DEVICE, weights_only=False)
model = build_model(
    best_name, input_dim=len(STATE_NAMES) + len(INPUT_NAMES), state_dim=len(STATE_NAMES),
    **checkpoint['model_kwargs'],
).to(DEVICE)
model.load_state_dict(checkpoint['model_state'])
normalization = Normalization.from_dict(checkpoint['normalization'])
with h5py.File(DATASET, 'r') as h5:
    truth = h5['test/states'][0, :1001]
    future_inputs = h5['test/inputs'][0, :1000]
prediction = rollout_trajectory(model, truth[0], future_inputs, normalization, DEVICE)
horizons_ms = (1, 5, 10, 25, 50, 100)
rollout_scores = pd.DataFrame([{
    'horizon_ms': horizon,
    'voltage_rmse_mV': float(np.sqrt(np.mean((prediction[:int(horizon / config.dt_ms) + 1, 0] - truth[:int(horizon / config.dt_ms) + 1, 0])**2))),
} for horizon in horizons_ms])
print('Autoregressive error growth for', best_name)
rollout_scores

In [ ]:
rollout_time = np.arange(len(truth)) * config.dt_ms
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(rollout_time, truth[:, 0], label='teacher', lw=1)
axes[0].plot(rollout_time, prediction[:, 0], label=best_name, lw=1, alpha=.8)
axes[0].set_ylabel('V (mV)'); axes[0].legend()
axes[1].plot(rollout_time, truth[:, 1] * 1e3, label='teacher')
axes[1].plot(rollout_time, prediction[:, 1] * 1e3, label=best_name, alpha=.8)
axes[1].set(xlabel='time (ms)', ylabel='Ca i (uM)'); axes[1].legend()
plt.tight_layout()

## Outputs

The HDF5 dataset, SHA-256 manifest, model checkpoints, and JSON metrics are in `hay_single_results`. Before drawing scientific conclusions, increase trajectory count and duration in `SimulationConfig`, keep the split policy unchanged, and report both one-step and rollout errors.